# Agent first pass: does interpreting the query bring the correct solution into view?

This notebook measures **only the first pass** of the planned agent, with no review loop yet:

1. An LLM (via API) reads each AppsRetrieval problem and writes:
   - **task**: the problem restated as a plain formal task, without the story;
   - **concepts**: 3 short search phrases (algorithm / technique / key operation);
   - **sketch**: a short Python code sketch of a typical solution.
2. Each of these is used as a separate search over the whole corpus with CodeRankEmbed (whole documents, no chunking, which is your best baseline).
3. For every query we record **where the correct solution ranks** in each search, in the combined (fused) ranking, and whether it lands in the **candidate pool** (top-K of any search) that an agent would review.

Everything is compared against the **original query** (your baseline) on the same queries.

Uses the same Drive folder as the main notebook. LLM outputs and embeddings are cached there, so a crash or re-run doesn't repeat paid API calls.

## Step 1: Install

In [ ]:
!pip install -q "mteb>=2.2" "sentence-transformers>=3.0" einops openai google-genai

## Step 2: Imports, GPU, Drive

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import json
import random
import re
import threading
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import mteb
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "no GPU")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    RUN_DIR = Path("/content/drive/MyDrive/code_retrieval_apps")
except ImportError:
    RUN_DIR = Path("code_retrieval_apps")
OUT_DIR = RUN_DIR / "agent_first_pass"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Saving to:", OUT_DIR.resolve())

## Step 3: Config
- `llm_provider`: `"openai"` or `"gemini"`, each with its own model setting. A small, fast model keeps the cost low for a first measurement; switch to a stronger one later to compare. Model names change often, so check the provider's model list if one stops working.
- `n_per_group`: queries sampled from each baseline group (good / mid / bad), so the bad group is well represented.
- `pool_ks`: how many results per search the agent would review. The "candidate pool" is the union of the top-K of every search.

In [ ]:
@dataclass(frozen=True)
class Config:
    # LLM: only "openai" or "gemini"
    llm_provider: str = "openai"

    # OpenAI
    openai_model: str = "gpt-5-mini"
    openai_reasoning_effort: str | None = "minimal"   # lowest reasoning = cheapest; None = model default
    openai_max_completion_tokens: int = 2000          # includes hidden reasoning tokens, so keep it higher
    openai_price_in: float | None = 0.25              # USD per 1M tokens, only for the cost estimate
    openai_price_out: float | None = 2.00             # (check OpenAI's pricing page)

    # Gemini
    gemini_model: str = "gemini-3.8-flash"
    gemini_max_output_tokens: int = 4000              # includes thinking tokens
    gemini_price_in: float | None = None              # fill in from Google's pricing page to get a cost estimate
    gemini_price_out: float | None = None

    llm_workers: int = 4               # parallel API calls
    n_concepts: int = 3

    # sample
    n_per_group: int = 100             # good / mid / bad -> 300 queries
    include_labelled: bool = True      # also include the queries from labelled_sheet
    seed: int = 0

    # cached vectors from the main notebook
    cache_tag: str = "cast-c512-anc1-imp0"      # the chunked main run ("no-chunking" = the ablation run)
    cache_version: str = "v1"
    labelled_sheet: str = "failure_labelling_sheet_chunked.csv"
    expected_baseline_ndcg: float = 0.2332      # nDCG@10 of that MTEB run, for the sanity check

    # embedding model (same settings as the main notebook)
    embed_model: str = "nomic-ai/CodeRankEmbed"
    query_prefix: str = "Represent this query for searching relevant code: "
    query_max_len: int = 2048
    doc_max_len: int = 1024
    query_batch: int = 16
    doc_batch: int = 64
    fp16: bool = True

    # evaluation
    pool_ks: tuple = (10, 20, 50)
    rrf_k: int = 60                    # standard constant for reciprocal rank fusion

    @property
    def llm_model(self) -> str:
        return self.openai_model if self.llm_provider == "openai" else self.gemini_model

CFG = Config()
assert CFG.llm_provider in ("openai", "gemini"), "llm_provider must be 'openai' or 'gemini'"
CFG

## Step 4: Load AppsRetrieval

In [ ]:
task = mteb.get_task("AppsRetrieval")
task.load_data()
subset = next(iter(task.dataset))
split = task.dataset[subset]["test"]

doc_ids = list(split["corpus"]["id"])
doc_texts = list(split["corpus"]["text"])
doc_index = {d: i for i, d in enumerate(doc_ids)}
query_text = dict(zip(split["queries"]["id"], split["queries"]["text"]))
qrels = split["relevant_docs"]
gold_of = {q: next(d for d, s in rel.items() if s > 0) for q, rel in qrels.items()}
all_qids = list(qrels)
print(f"{len(doc_ids)} documents | {len(all_qids)} queries")

## Step 5: Load the document and original-query vectors from the main notebook's cache
Loads the vectors your MTEB run saved (set by `cache_tag`; by default the chunked main run, where each document vector is the pooled vector of its AST chunks). Nothing is re-embedded here; the model is still loaded because Step 8 needs it for the new searches. Then the baseline rank of the correct solution is computed over the **entire corpus** for every query. As a sanity check, the baseline nDCG@10 should match that MTEB run (`expected_baseline_ndcg`).

In [ ]:
model = SentenceTransformer(CFG.embed_model, trust_remote_code=True, device=DEVICE)
if CFG.fp16 and DEVICE == "cuda":
    model.half()

def embed(texts, is_query: bool) -> np.ndarray:
    # is_query=True -> natural-language search text (gets the prefix); False -> code
    model.max_seq_length = CFG.query_max_len if is_query else CFG.doc_max_len
    if is_query:
        texts = [CFG.query_prefix + t for t in texts]
    emb = model.encode(texts, batch_size=CFG.query_batch if is_query else CFG.doc_batch,
                       normalize_embeddings=True, convert_to_numpy=True,
                       show_progress_bar=len(texts) > 200)
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return emb.astype(np.float32)

from mteb.models.cache_wrappers.cache_backends import NumpyCache

def load_mteb_cache(kind: str, texts: list[str]) -> np.ndarray:
    # kind = "document" or "query". MTEB finds each vector by the exact text it embedded.
    folder = RUN_DIR / "embedding_cache" / CFG.cache_version / CFG.cache_tag / "AppsRetrieval" / kind
    if not (folder / "index.json").exists():
        raise FileNotFoundError(f"No MTEB cache at {folder}. Check cache_tag / cache_version.")
    cache = NumpyCache(folder)
    cache.load()
    vecs, missing = [], 0
    for t in texts:
        v = cache.get_vector({"text": t})
        if v is None:
            v = cache.get_vector({"text": t.strip()})   # MTEB strips document text before embedding
        if v is None:
            missing += 1
        else:
            vecs.append(np.asarray(v, dtype=np.float32))
    if missing:
        raise RuntimeError(f"{missing} of {len(texts)} {kind} texts not found in {folder}. "
                           "The cache may come from a different run.")
    print(f"{kind}: {len(vecs)} vectors loaded from '{CFG.cache_tag}'")
    return np.stack(vecs)

D = load_mteb_cache("document", doc_texts)                           # (n_docs, dim)
Q = load_mteb_cache("query", [query_text[q] for q in all_qids])     # original queries

def rank_of_gold(scores: np.ndarray, gold_idx: int) -> int:
    # 1 = best. Rank over the whole corpus.
    return int((scores > scores[gold_idx]).sum()) + 1

base_scores = Q @ D.T
baseline_rank = {q: rank_of_gold(base_scores[i], doc_index[gold_of[q]]) for i, q in enumerate(all_qids)}
del base_scores

def ndcg10(r): return 1 / np.log2(r + 1) if r <= 10 else 0.0   # one relevant doc per query
print(f"Baseline over all queries: nDCG@10 = {np.mean([ndcg10(r) for r in baseline_rank.values()]):.4f} "
      f"(MTEB run '{CFG.cache_tag}': {CFG.expected_baseline_ndcg})")

## Step 6: Pick the sample (stratified by baseline rank, fixed after the first run)

In [ ]:
def group_of(r): return "good (1-10)" if r <= 10 else "mid (11-100)" if r <= 100 else "bad (>100)"

sample_path = OUT_DIR / "sample_queries.json"
if sample_path.exists():
    sample = json.loads(sample_path.read_text())       # same queries every run -> LLM answers are reused
    print(f"Loaded fixed sample from {sample_path}")
else:
    by_group = {}
    for q in all_qids:
        by_group.setdefault(group_of(baseline_rank[q]), []).append(q)
    rng = random.Random(CFG.seed)
    sample = []
    for g in ["good (1-10)", "mid (11-100)", "bad (>100)"]:
        sample += rng.sample(by_group[g], min(CFG.n_per_group, len(by_group[g])))
    sheet_path = RUN_DIR / CFG.labelled_sheet
    if CFG.include_labelled and sheet_path.exists():
        extra = [q for q in pd.read_csv(sheet_path)["qid"] if q in qrels and q not in sample]
        sample += extra
        print(f"Added {len(extra)} queries from {CFG.labelled_sheet}")
    sample_path.write_text(json.dumps(sample))
    print(f"Saved fixed sample to {sample_path}")

print(f"Sample: {len(sample)} queries |", dict(Counter(group_of(baseline_rank[q]) for q in sample)))

## Step 7: The LLM interpretation step
The API key is read from Colab's **Secrets** panel (key icon on the left): add `OPENAI_API_KEY` or `GEMINI_API_KEY` (matching `llm_provider`) and allow notebook access. Never paste the key into the notebook itself.

Each result is appended to a JSONL file on Drive as soon as it arrives, so re-running this cell only calls the API for queries that aren't done yet. Token usage is recorded so you can estimate cost with the provider's pricing page.

Both providers get exactly the same instructions, so results are comparable. The interpretations are cached per model name, so switching provider or model never mixes results.

In [ ]:
def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

if CFG.llm_provider == "openai":
    from openai import OpenAI
    client = OpenAI(api_key=secret("OPENAI_API_KEY"), max_retries=5)   # retries rate limits itself
else:
    from google import genai
    from google.genai import types as genai_types
    client = genai.Client(api_key=secret("GEMINI_API_KEY"))
print(f"LLM: {CFG.llm_provider} / {CFG.llm_model}")

SYSTEM_PROMPT = f'''You help a code search engine find Python solutions to competitive programming problems.
The engine compares your text with the solutions' code, so the story in the problem (names, places, characters) does not help.

Read the problem and return ONLY a JSON object with these keys:
- "task": the problem restated in 1-2 plain sentences as a formal computational task, without the story.
- "concepts": a list of exactly {CFG.n_concepts} short search phrases (3-10 words each). Each should name a
  likely algorithm, technique or key operation of a solution (e.g. "binary search on sorted array",
  "count characters and take minimum ratio"). Make them different from each other.
- "sketch": a short Python solution sketch (at most 15 lines) in typical competitive programming style,
  reading input with input(). It does not have to be fully correct.

Return only the JSON, with no extra text.'''

def call_llm(prompt: str):
    # Returns (text, input_tokens, output_tokens). Output tokens include hidden reasoning/thinking tokens.
    if CFG.llm_provider == "openai":
        kwargs = dict(
            model=CFG.openai_model,
            messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt}],
            max_completion_tokens=CFG.openai_max_completion_tokens,
            response_format={"type": "json_object"},        # forces valid JSON
        )
        if CFG.openai_reasoning_effort:
            kwargs["reasoning_effort"] = CFG.openai_reasoning_effort
        resp = client.chat.completions.create(**kwargs)
        text = resp.choices[0].message.content or ""
        if not text:
            raise RuntimeError(f"empty answer (finish_reason={resp.choices[0].finish_reason}); "
                               "raise openai_max_completion_tokens or lower the reasoning effort")
        return text, resp.usage.prompt_tokens, resp.usage.completion_tokens

    # Gemini: retry a few times on errors such as rate limits
    config = genai_types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        max_output_tokens=CFG.gemini_max_output_tokens,
        response_mime_type="application/json",              # forces JSON output
    )
    for attempt in range(5):
        try:
            resp = client.models.generate_content(model=CFG.gemini_model, contents=prompt, config=config)
            break
        except Exception:
            if attempt == 4:
                raise
            time.sleep(2 ** (attempt + 1))
    text = resp.text or ""
    if not text:
        raise RuntimeError("empty answer; raise gemini_max_output_tokens")
    usage = resp.usage_metadata
    tin = getattr(usage, "prompt_token_count", 0) or 0
    tout = (getattr(usage, "candidates_token_count", 0) or 0) + (getattr(usage, "thoughts_token_count", 0) or 0)
    return text, tin, tout

def parse_interpretation(text: str) -> dict:
    # Take the outermost {...} (models sometimes add ```json fences) and check the fields.
    start, end = text.find("{"), text.rfind("}")
    obj = json.loads(text[start:end + 1])
    concepts = [str(c).strip() for c in obj.get("concepts", []) if str(c).strip()][:CFG.n_concepts]
    return {"task": str(obj.get("task", "")).strip(), "concepts": concepts,
            "sketch": str(obj.get("sketch", "")).strip()}

cache_path = OUT_DIR / f"interpretations_{CFG.llm_model}.jsonl"
interp = {}
if cache_path.exists():
    for line in cache_path.read_text().splitlines():
        rec = json.loads(line)
        if rec.get("ok"):
            interp[rec["qid"]] = rec
print(f"Cached interpretations: {len(interp)}")

write_lock = threading.Lock()
def interpret(qid):
    rec = {"qid": qid, "model": CFG.llm_model}
    try:
        text, tin, tout = call_llm(query_text[qid])
        rec.update(parse_interpretation(text), ok=True, input_tokens=tin, output_tokens=tout)
    except Exception as e:                       # keep going; failed queries are retried on the next run
        rec.update(ok=False, error=f"{type(e).__name__}: {e}"[:300])
    with write_lock, open(cache_path, "a") as f:
        f.write(json.dumps(rec) + "\n")
    return rec

todo = [q for q in sample if q not in interp]
failed = []
with ThreadPoolExecutor(max_workers=CFG.llm_workers) as pool:
    for rec in tqdm(as_completed([pool.submit(interpret, q) for q in todo]), total=len(todo), desc="LLM"):
        rec = rec.result()
        if rec["ok"]:
            interp[rec["qid"]] = rec
        else:
            failed.append(rec)

done = [q for q in sample if q in interp]
print(f"Interpreted: {len(done)}/{len(sample)} | failed this run: {len(failed)}")
if failed:
    print("Example error:", failed[0]["error"])
tin = sum(interp[q]["input_tokens"] for q in done)
tout = sum(interp[q]["output_tokens"] for q in done)
print(f"Tokens for the sample: {tin:,} input + {tout:,} output "
      f"(avg {tin / max(len(done), 1):.0f} + {tout / max(len(done), 1):.0f} per query)")
price_in, price_out = ((CFG.openai_price_in, CFG.openai_price_out) if CFG.llm_provider == "openai"
                        else (CFG.gemini_price_in, CFG.gemini_price_out))
if price_in is not None and price_out is not None:
    cost = tin / 1e6 * price_in + tout / 1e6 * price_out
    print(f"Estimated cost at ${price_in}/${price_out} per 1M tokens: ${cost:.2f} for the sample, "
          f"~${cost / max(len(done), 1) * len(all_qids):.2f} for all {len(all_qids)} queries")
else:
    print("Set the price fields in Config to get a cost estimate.")

## Step 8: Run every search and record where the correct solution lands
For each query we score the whole corpus with each search text separately, then combine them:

| Name | What it is |
|---|---|
| `original` | the original problem text (your baseline) |
| `task` | the LLM's formal restatement |
| `concept_1..3` | each search phrase |
| `sketch` | the code sketch, searched as **code** (code-to-code, no query prefix) |
| `rrf_interp` | reciprocal rank fusion of task + concepts + sketch |
| `rrf_all` | the same plus the original query |
| `mean_all` | average cosine score over all searches (same as averaging the search vectors) |

**Candidate pool** = union of the top-K results of the interpretation searches (task, concepts, sketch), i.e. what an agent would review. `pool+orig` also adds the original query's top-K.

In [ ]:
def ranks_of_all(scores: np.ndarray) -> np.ndarray:
    # rank (1 = best) of every document for one search
    order = np.argsort(-scores)
    ranks = np.empty_like(order)
    ranks[order] = np.arange(1, len(scores) + 1)
    return ranks

rows, pool_rows = [], []
q_index = {q: i for i, q in enumerate(all_qids)}
for qid in tqdm(done, desc="searching"):
    it = interp[qid]
    gold_idx = doc_index[gold_of[qid]]

    nl_texts = {"task": it["task"], **{f"concept_{i + 1}": c for i, c in enumerate(it["concepts"])}}
    nl_emb = embed(list(nl_texts.values()), is_query=True)
    searches = {"original": Q[q_index[qid]], **dict(zip(nl_texts, nl_emb))}
    if it["sketch"]:
        searches["sketch"] = embed([it["sketch"]], is_query=False)[0]

    scores = {name: D @ vec for name, vec in searches.items()}
    ranks = {name: ranks_of_all(s) for name, s in scores.items()}
    interp_names = [n for n in scores if n != "original"]

    def rrf(names):
        return sum(1.0 / (CFG.rrf_k + ranks[n]) for n in names)
    fused = {
        "rrf_interp": rrf(interp_names),
        "rrf_all": rrf(list(scores)),
        "mean_all": np.mean([scores[n] for n in scores], axis=0),
    }

    row = {"qid": qid, "group": group_of(baseline_rank[qid]), "baseline_rank": baseline_rank[qid]}
    for name, s in {**scores, **fused}.items():
        row[name] = rank_of_gold(s, gold_idx)
    rows.append(row)

    prow = {"qid": qid, "group": row["group"]}
    for k in CFG.pool_ks:
        # "in pool" uses the same gold ranks as the tables (ties count in the gold's favour,
        # which matters because APPS contains identical solutions with identical scores)
        in_pool = any(row[n] <= k for n in interp_names)
        pool = set().union(*(np.where(ranks[n] <= k)[0] for n in interp_names))
        pool_orig = pool | set(np.where(ranks["original"] <= k)[0])
        prow[f"in_pool@{k}"], prow[f"pool_size@{k}"] = in_pool, len(pool)
        prow[f"in_pool+orig@{k}"], prow[f"pool+orig_size@{k}"] = in_pool or row["original"] <= k, len(pool_orig)
    pool_rows.append(prow)

res = pd.DataFrame(rows)
pool_df = pd.DataFrame(pool_rows)
res.merge(pool_df, on=["qid", "group"]).merge(
    pd.DataFrame([{"qid": q, **{k: interp[q][k] for k in ("task", "concepts", "sketch")}} for q in done]),
    on="qid").to_csv(OUT_DIR / f"first_pass_results_{CFG.llm_model}_{CFG.cache_tag}.csv", index=False)
print("Saved per-query results to", OUT_DIR / f"first_pass_results_{CFG.llm_model}_{CFG.cache_tag}.csv")

## Step 9: Results
**Note:** the sample is stratified (one third each of good / mid / bad), so the overall numbers are *not* comparable to the full-benchmark scores. Compare the rows with each other, and look at the per-group tables.

In [ ]:
methods = ["original", "task"] + [f"concept_{i + 1}" for i in range(CFG.n_concepts)] + \
          ["sketch", "rrf_interp", "rrf_all", "mean_all"]
methods = [m for m in methods if m in res]

def summarize(df):
    out = {}
    for m in methods:
        r = df[m].dropna()
        out[m] = {
            "nDCG@10": np.mean([ndcg10(x) for x in r]),
            "R@10": (r <= 10).mean(),
            "R@20": (r <= 20).mean(),
            "R@100": (r <= 100).mean(),
            "median rank": r.median(),
        }
    return pd.DataFrame(out).T

print("All sampled queries:")
print(summarize(res).round(3).to_string(), "\n")

groups = ["good (1-10)", "mid (11-100)", "bad (>100)"]
print("Median rank of the correct solution, per baseline group:")
print(pd.DataFrame({g: res[res["group"] == g][methods].median() for g in groups}).round(0).to_string(), "\n")

print("R@100 (correct solution in the top 100), per baseline group:")
print(pd.DataFrame({g: (res[res["group"] == g][methods] <= 100).mean() for g in groups}).round(3).to_string())

In [ ]:
print("Candidate pool: is the correct solution among the results an agent would review?")
pool_table = {}
for k in CFG.pool_ks:
    for label, col, size_col in [("interpretations", f"in_pool@{k}", f"pool_size@{k}"),
                                 ("+ original", f"in_pool+orig@{k}", f"pool+orig_size@{k}")]:
        pool_table[f"top-{k} per search, {label}"] = {
            **{g: pool_df[pool_df["group"] == g][col].mean() for g in groups},
            "all": pool_df[col].mean(),
            "avg pool size": pool_df[size_col].mean(),
        }
print(pd.DataFrame(pool_table).T.round(3).to_string(), "\n")

# How many bad-group queries does the first pass rescue?
bad = res[res["group"] == "bad (>100)"]
if len(bad):
    best = bad[["task"] + [m for m in methods if m.startswith(("concept", "sketch"))]].min(axis=1)
    print(f"Bad-group queries ({len(bad)}): correct solution now in top 100 by rrf_all: {(bad['rrf_all'] <= 100).mean():.1%}, "
          f"in top 20 of at least one interpretation search: {(best <= 20).mean():.1%}")

### Results by failure cause
Uses the causes you labelled in the labelling sheet. Only a handful of queries per cause, so read it as hints, not proof.

In [ ]:
sheet_file = RUN_DIR / CFG.labelled_sheet
if sheet_file.exists():
    labels = pd.read_csv(sheet_file)[["qid", "cause"]].dropna()
    lab = res.merge(labels, on="qid")
    print("Does the first pass help, per failure cause?")
    print(lab.groupby("cause").agg(
        queries=("qid", "size"),
        baseline_median=("baseline_rank", "median"),
        rrf_all_median=("rrf_all", "median"),
        sketch_median=("sketch", "median"),
        top100_rrf_all=("rrf_all", lambda r: (r <= 100).mean()),
        top100_sketch=("sketch", lambda r: (r <= 100).mean()),
    ).round(2).to_string())
else:
    print(f"No labelling sheet found at {sheet_file}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = {"good (1-10)": "tab:green", "mid (11-100)": "tab:orange", "bad (>100)": "tab:red"}
for ax, m in zip(axes, ["rrf_all", "sketch"]):
    if m not in res:
        continue
    for g, c in colors.items():
        d = res[res["group"] == g]
        ax.scatter(d["baseline_rank"], d[m], s=12, alpha=0.6, c=c, label=g)
    lim = [1, len(doc_ids)]
    ax.plot(lim, lim, "k--", lw=1)
    ax.axhline(100, color="grey", lw=0.8, ls=":")
    ax.set(xscale="log", yscale="log", xlabel="baseline rank (original query)", ylabel=f"rank with {m}",
           title=f"{m}: below the diagonal = improved")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Step 10: Look at individual queries
Shows the LLM's interpretation and where the correct solution ranked in each search. By default: the 3 biggest improvements and 3 queries that stayed bad. Put your own query IDs (e.g. from the labelling sheet) in `SHOW_QIDS` to inspect them.

In [ ]:
SHOW_QIDS = []          # e.g. ["q8364", "q5764"]

def show(qid):
    r = res.set_index("qid").loc[qid]
    it = interp[qid]
    print("=" * 100)
    print(f"{qid} | {r['group']} | gold {gold_of[qid]}")
    print("PROBLEM:", " ".join(query_text[qid].split())[:300], "...")
    print("TASK:   ", it["task"])
    for i, c in enumerate(it["concepts"]):
        print(f"CONCEPT {i + 1}:", c)
    print("SKETCH:\n   " + it["sketch"].replace("\n", "\n   "))
    print("RANK OF CORRECT SOLUTION:", " | ".join(f"{m} {int(r[m])}" for m in methods if m in r and not pd.isna(r[m])))
    print("CORRECT SOLUTION (start):\n   " + doc_texts[doc_index[gold_of[qid]]][:400].replace("\n", "\n   "))

res["gain"] = np.log10(res["baseline_rank"]) - np.log10(res["rrf_all"])
picks = SHOW_QIDS or (list(res.nlargest(3, "gain")["qid"]) +
                      list(res[(res["group"] == "bad (>100)") & (res["rrf_all"] > 100)].sample(
                          min(3, int(((res["group"] == "bad (>100)") & (res["rrf_all"] > 100)).sum())),
                          random_state=0)["qid"]))
for q in picks:
    show(q)

## Step 11: Review step: run the candidates on the example tests

For each query, the agent reviews a **pool** = top-K of the **sketch** search + top-K of the **original** query. Every candidate in the pool is run on the example inputs from the problem statement, and its output is compared with the expected output. No LLM calls: this step is free and not affected by memorization.

**Final ranking:** candidates that pass all examples first, then everything else in the order of the **base ranking** (average of the sketch and original scores).

### Step 11a: Settings and reading the examples from the problem text
Handles the two APPS formats: `-----Examples----- Input … Output …` (Codeforces style) and `-----Sample Input----- … -----Sample Output-----` (AtCoder style). Problems without readable examples (e.g. LeetCode-style "write a function" problems) are not reviewed and keep their base ranking.

In [ ]:
import math
import subprocess
import sys
import tempfile
import hashlib

@dataclass(frozen=True)
class ReviewConfig:
    pool_k: int = 20                   # top-K from the sketch search + top-K from the original query
    timeout_s: float = 4.0             # per example run
    mem_mb: int = 1024                 # memory limit per run
    workers: int = max(2, os.cpu_count() or 2)
    float_tol: float = 1e-6            # allowed difference for decimal numbers in outputs

RCFG = ReviewConfig()

SAMPLE_RE = re.compile(r"-----Sample Input[^\n]*?-----\n(.*?)\n-----Sample Output[^\n]*?-----\n(.*?)(?=\n-----|\Z)", re.S)
EXAMPLES_RE = re.compile(r"-----Examples?-----\n(.*?)(?=\n-----|\Z)", re.S)

def parse_examples(problem: str) -> list[tuple[str, str]]:
    # Returns [(input, expected_output), ...] or [] if none can be read.
    # AtCoder style: an explanation can follow the output after a blank line, so keep only the first block
    exs = [(i.strip("\n") + "\n", o.strip().split("\n\n")[0].strip()) for i, o in SAMPLE_RE.findall(problem)]
    if exs:
        return exs
    m = EXAMPLES_RE.search(problem)
    if not m:
        return []
    parts = re.split(r"^(Input|Output)\s*$", m.group(1), flags=re.M)
    cur_in = None
    for tag, text in zip(parts[1::2], parts[2::2]):
        if tag == "Input":
            cur_in = text.strip("\n") + "\n"
        elif cur_in is not None:
            exs.append((cur_in, text.strip()))
            cur_in = None
    return [(i, o) for i, o in exs if o]

examples = {q: parse_examples(query_text[q]) for q in done}
with_ex = [q for q in done if examples[q]]
print(f"Queries with readable examples: {len(with_ex)} of {len(done)} ({len(with_ex) / len(done):.1%})")
print(f"Examples per query: mean {np.mean([len(examples[q]) for q in with_ex]):.2f}")
print("Per group:", {g: f"{np.mean([bool(examples[q]) for q in done if group_of(baseline_rank[q]) == g]):.0%}"
                     for g in ["good (1-10)", "mid (11-100)", "bad (>100)"]})

q_demo = with_ex[0]
print(f"\nExample from {q_demo}:")
for i, (inp, out) in enumerate(examples[q_demo][:2]):
    print(f"--- input {i + 1} ---\n{inp}--- expected output ---\n{out}")

### Step 11b: The runner, and how reliable the check is
Runs a solution with `python3` in a separate process, with a time and memory limit, in an empty temporary folder. Output comparison ignores extra whitespace, allows small differences in decimal numbers, and treats `YES`/`yes` as equal (Codeforces usually accepts either case).

Then it runs the **correct solution** of every sampled query on its own examples. That pass rate is the ceiling of the review: when the correct solution fails its own examples (several valid answers, Python 2 code, etc.), the review can't recognise it.

In [ ]:
import resource

def _limit_resources():
    lim = RCFG.mem_mb * 2**20
    resource.setrlimit(resource.RLIMIT_AS, (lim, lim))

def outputs_match(got: str, expected: str) -> bool:
    g, e = got.split(), expected.split()
    if g == e:
        return True
    if len(g) != len(e):
        return False
    for a, b in zip(g, e):
        if a == b or a.lower() == b.lower():
            continue
        try:
            if not math.isclose(float(a), float(b), rel_tol=RCFG.float_tol, abs_tol=RCFG.float_tol):
                return False
        except ValueError:
            return False
    return True

def run_on_examples(code: str, exs) -> str:
    # "pass" if all examples pass, otherwise the first problem: "wrong", "error" or "timeout".
    with tempfile.TemporaryDirectory() as d:
        path = Path(d) / "solution.py"
        path.write_text(code, encoding="utf-8")
        for inp, expected in exs:
            try:
                p = subprocess.run([sys.executable, str(path)], input=inp, capture_output=True,
                                   text=True, encoding="utf-8", errors="replace",
                                   timeout=RCFG.timeout_s, cwd=d, preexec_fn=_limit_resources)
            except subprocess.TimeoutExpired:
                return "timeout"
            if p.returncode != 0:
                return "error"
            if not outputs_match(p.stdout, expected):
                return "wrong"
    return "pass"

# ---- cache of verdicts on Drive: one line per (query, document) ----
verdict_path = OUT_DIR / "review_verdicts.jsonl"
verdicts = {}
if verdict_path.exists():
    for line in verdict_path.read_text().splitlines():
        r = json.loads(line)
        verdicts[(r["qid"], r["doc"])] = r["verdict"]
print(f"Cached verdicts: {len(verdicts)}")
verdict_lock = threading.Lock()

def review_pairs(pairs, desc):
    # pairs = [(qid, doc_id), ...]; runs the ones not cached yet, in parallel
    todo = [p for p in dict.fromkeys(pairs) if p not in verdicts]
    def job(pair):
        qid, doc = pair
        v = run_on_examples(doc_texts[doc_index[doc]], examples[qid])
        with verdict_lock:
            verdicts[pair] = v
            with open(verdict_path, "a") as f:
                f.write(json.dumps({"qid": qid, "doc": doc, "verdict": v}) + "\n")
    with ThreadPoolExecutor(max_workers=RCFG.workers) as pool:
        list(tqdm(pool.map(job, todo), total=len(todo), desc=desc))

# ---- reliability: does the correct solution pass its own examples? ----
review_pairs([(q, gold_of[q]) for q in with_ex], "correct solutions")
gold_verdict = Counter(verdicts[(q, gold_of[q])] for q in with_ex)
print("\nCorrect solution on its own examples:", dict(gold_verdict))
print(f"Correct-solution pass rate: {gold_verdict['pass'] / len(with_ex):.1%}  <- the review can only rescue these")

### Step 11c: Build the pools and run every candidate
Pool = top-K of the sketch search + top-K of the original query. Verdicts are saved to Drive as they finish; if the run stops, re-running this cell continues where it left off.

In [ ]:
sketch_qids = [q for q in done if interp[q]["sketch"]]
S = embed([interp[q]["sketch"] for q in sketch_qids], is_query=False)
sketch_vec = dict(zip(sketch_qids, S))

review = {}
for qid in done:
    orig = D @ Q[q_index[qid]]
    if qid in sketch_vec:
        sk = D @ sketch_vec[qid]
        base = (sk + orig) / 2
        pool = set(np.argsort(-sk)[:RCFG.pool_k]) | set(np.argsort(-orig)[:RCFG.pool_k])
    else:
        base = orig
        pool = set(np.argsort(-orig)[:RCFG.pool_k])
    review[qid] = {"base": base, "pool": sorted(int(i) for i in pool)}

pairs = [(q, doc_ids[i]) for q in with_ex for i in review[q]["pool"]]
print(f"Pools: {len(with_ex)} queries x avg {len(pairs) / max(len(with_ex), 1):.1f} candidates = {len(pairs)} runs")
t0 = time.time()
review_pairs(pairs, "candidates")
print(f"Done in {(time.time() - t0) / 60:.1f} min")

### Step 11d: Results after review
`base` = average of sketch and original scores (no review). `reviewed` = base, but candidates that passed the examples moved to the top (keeping their base order). The group tables compare every method on the same queries.

In [ ]:
rows = []
for qid in done:
    d = review[qid]
    gold_idx = doc_index[gold_of[qid]]
    final = d["base"].copy()
    passed = [i for i in d["pool"] if verdicts.get((qid, doc_ids[i])) == "pass"] if qid in examples and examples[qid] else []
    final[passed] += 10.0                              # scores are <= 1, so passers go above everything else
    r = res.set_index("qid").loc[qid]
    rows.append({
        "qid": qid, "group": r["group"], "has_examples": bool(examples[qid]),
        "original": int(r["original"]), "sketch": int(r["sketch"]) if not pd.isna(r.get("sketch")) else np.nan,
        "base": rank_of_gold(d["base"], gold_idx), "reviewed": rank_of_gold(final, gold_idx),
        "gold_in_pool": gold_idx in d["pool"], "n_passed": len(passed),
        "gold_passed": gold_idx in passed, "wrong_passed": sum(i != gold_idx for i in passed),
    })
rev = pd.DataFrame(rows)
rev.to_csv(OUT_DIR / f"review_results_{CFG.llm_model}_{CFG.cache_tag}.csv", index=False)

m_cols = ["original", "sketch", "base", "reviewed"]
def table(df):
    return pd.DataFrame({m: {"nDCG@10": np.mean([ndcg10(x) for x in df[m].dropna()]),
                             "R@1": (df[m] <= 1).mean(), "R@10": (df[m] <= 10).mean(),
                             "R@100": (df[m] <= 100).mean(), "median rank": df[m].median()}
                         for m in m_cols}).T.round(3)

print(f"All sampled queries ({len(rev)}):")
print(table(rev).to_string(), "\n")
print(f"Only queries with readable examples ({int(rev['has_examples'].sum())}):")
print(table(rev[rev["has_examples"]]).to_string(), "\n")

groups = ["good (1-10)", "mid (11-100)", "bad (>100)"]
print("Median rank per baseline group:")
print(pd.DataFrame({g: rev[rev["group"] == g][m_cols].median() for g in groups}).to_string(), "\n")
print("R@1 per baseline group:")
print(pd.DataFrame({g: (rev[rev["group"] == g][m_cols] <= 1).mean() for g in groups}).round(3).to_string(), "\n")

ex = rev[rev["has_examples"]]
print("How the review behaved (queries with examples):")
print(f"  correct solution in the pool:                 {ex['gold_in_pool'].mean():.1%}")
print(f"  ... and it passed the examples:               {ex['gold_passed'].mean():.1%}")
print(f"  queries where >=1 WRONG candidate passed:     {(ex['wrong_passed'] > 0).mean():.1%}")
print(f"  passing candidates per query: mean {ex['n_passed'].mean():.2f} | "
      f"no candidate passed: {(ex['n_passed'] == 0).mean():.1%}")

In [ ]:
def show_review(qid, max_code=400, max_shown=4):
    r = rev.set_index("qid").loc[qid]
    d = review[qid]
    print("=" * 100)
    print(f"{qid} | {r['group']} | ranks: original {r['original']} -> base {r['base']} -> reviewed {r['reviewed']} "
          f"| passed: {r['n_passed']} (wrong: {r['wrong_passed']})")
    print("PROBLEM:", " ".join(query_text[qid].split())[:250], "...")
    order = sorted(d["pool"], key=lambda i: -d["base"][i])
    passers = [i for i in order if verdicts.get((qid, doc_ids[i])) == "pass"][:max_shown]
    for i in passers:
        tag = "CORRECT" if doc_ids[i] == gold_of[qid] else "wrong"
        print(f"\n  passed: {doc_ids[i]} ({tag})\n   " + doc_texts[i][:max_code].replace("\n", "\n   "))
    if r["n_passed"] > len(passers):
        print(f"\n  ... and {r['n_passed'] - len(passers)} more passed")

rescued = rev[(rev["base"] > 10) & (rev["reviewed"] == 1)]
false_pass = rev[rev["has_examples"] & (rev["wrong_passed"] > 0) & (rev["reviewed"] > 1)]   # a wrong passer is above the correct one
print(f"Rescued (base rank > 10 -> rank 1): {len(rescued)} | wrong solutions ranked above the correct one: {len(false_pass)}")
for q in rescued.sample(min(2, len(rescued)), random_state=0)["qid"]:
    show_review(q)
for q in false_pass.sample(min(2, len(false_pass)), random_state=0)["qid"]:
    show_review(q)